In [ ]:
## Repo github Crypto gisaza https://github.com/gisazae/cryptographyOps_PythonJsJava

In [ ]:
## Multiple Algorithms for CryptoOps [Sym, Asym, Hash, DSA, EllipticCurve]

In [ ]:
!pip3 install -U PyCryptodome

In [ ]:
!ls

sample_data


In [ ]:
from Crypto.Util import number
number.getPrime(N=1024, randfunc=None)

149207604810665491613962505980266686464564171108142259493634046878500221787064349821403403787556761794566040935255765016927180824327482119231656002067602611790139322230528172377699258206715201021078361668142699320127864923583421546518062795827975449002830980220197322018983248753534114979686969105240374105921

In [ ]:
from Crypto.Hash import SHA256
h = SHA256.new()
h.update(b'Prueba11')
print(h.hexdigest())

2525264180aaab79ca289e72d14808a74628443cd03fbed52a74b473e0448eea


In [ ]:
from Crypto.Cipher import DES
des = DES.new('01234567'.encode("utf8"), DES.MODE_ECB)
text = 'prueba12'
cipher_text = des.encrypt(text.encode("utf8"))
print(cipher_text)
des.decrypt(cipher_text)

b'\x16\xa5\x1a\xcf\x19\xc63k'


b'prueba12'

In [ ]:
# Crypto with Sym DES mode CFB with IV
from Crypto.Cipher import DES
from Crypto import Random
iv = Random.get_random_bytes(8)
des1 = DES.new('01234567'.encode("utf8"), DES.MODE_CFB, iv)
des2 = DES.new('01234567'.encode("utf8"), DES.MODE_CFB, iv)
text = 'abcdefghijklmnop'
cipher_text = des1.encrypt(text.encode("utf8"))
print(cipher_text)
des2.decrypt(cipher_text)

b'\x93W\xb7\x92\x97\x1dJ\xc7\xee\xb3\xa6\x1e\xee\xba\x05['


b'abcdefghijklmnop'

In [ ]:
# Symetric with AES
from Crypto.Cipher import AES
obj = AES.new('This is a keyabc'.encode("utf8"), AES.MODE_CBC, 'This is an IV456'.encode("utf8"))
message = "Text Plain Test1"
ciphertext = obj.encrypt(message.encode("utf8"))
print(ciphertext)
obj2 = AES.new('This is a keyabc'.encode("utf8"), AES.MODE_CBC, 'This is an IV456'.encode("utf8"))
obj2.decrypt(ciphertext)

b'e[e\xe3\x82\x82W\xe9U\x1a\x8f\x8b\xab3%\xc4'


b'Text Plain Test1'

In [ ]:
from Crypto.Cipher import ARC4
obj1 = ARC4.new('01234567'.encode("utf8"))
obj2 = ARC4.new('01234567'.encode("utf8"))
text = 'abcdefghijklmnop'
cipher_text = obj1.encrypt(text.encode("utf8"))
print(cipher_text)
obj2.decrypt(cipher_text)


b'\xf0\xb7\x90{#ABXY9\xd06\x9f\xc0\x8c '


b'abcdefghijklmnop'

In [ ]:
#Generate Pub and Priv key
from Crypto.PublicKey import RSA

key = RSA.generate(2048)
private_key = key.export_key()
file_out = open("private.pem", "wb")
file_out.write(private_key)
file_out.close()

public_key = key.publickey().export_key()
file_out = open("receiver.pem", "wb")
file_out.write(public_key)
file_out.close()

In [ ]:
"""cifra una pieza de datos para un receptor del que tenemos la clave pública RSA.
La clave pública RSA se almacena en un archivo llamado receiver.pem.

Dado que queremos poder cifrar una cantidad arbitraria de datos, se usa un esquema de cifrado híbrido.
Usamos RSA con PKCS#1 OAEP para el cifrado asimétrico de una clave de sesión AES.
La clave de sesión se puede usar para cifrar todos los datos reales.

el modo EAX para permitir la detección de modificaciones no autorizadas."""

from Crypto.PublicKey import RSA
from Crypto.Random import get_random_bytes
from Crypto.Cipher import AES, PKCS1_OAEP

data = "Conozco alinenígenas".encode("utf-8")
file_out = open("encrypted_data.bin", "wb")

recipient_key = RSA.import_key(open("receiver.pem").read())
session_key = get_random_bytes(16)

# Encrypt the session key with the public RSA key
cipher_rsa = PKCS1_OAEP.new(recipient_key)
enc_session_key = cipher_rsa.encrypt(session_key)

# Encrypt the data with the AES session key
cipher_aes = AES.new(session_key, AES.MODE_EAX)
ciphertext, tag = cipher_aes.encrypt_and_digest(data)
[ file_out.write(x) for x in (enc_session_key, cipher_aes.nonce, tag, ciphertext) ]
file_out.close()

In [ ]:
!ls

encrypted_data.bin  private.pem  receiver.pem  sample_data


In [ ]:
"""El receptor tiene la clave RSA privada. Primero lo usarán para descifrar la clave de sesión, y con eso el resto del archivo"""
from Crypto.PublicKey import RSA
from Crypto.Cipher import AES, PKCS1_OAEP

file_in = open("encrypted_data.bin", "rb")

private_key = RSA.import_key(open("private.pem").read())

enc_session_key, nonce, tag, ciphertext = \
   [ file_in.read(x) for x in (private_key.size_in_bytes(), 16, 16, -1) ]
file_in.close()

# Decrypt the session key with the private RSA key
cipher_rsa = PKCS1_OAEP.new(private_key)
session_key = cipher_rsa.decrypt(enc_session_key)

# Decrypt the data with the AES session key
cipher_aes = AES.new(session_key, AES.MODE_EAX, nonce)
data = cipher_aes.decrypt_and_verify(ciphertext, tag)
print(data.decode("utf-8"))

Conozco alinenígenas


In [ ]:
from Crypto.Hash import SHA256
from Crypto.PublicKey import RSA
msg = b'A message for signing'
from Crypto.PublicKey import RSA
keyPair = RSA.generate(bits=1024)
print(f"Public key: (n={hex(keyPair.n)}, e={hex(keyPair.e)})")
print(f"Private key: (n={hex(keyPair.n)}, d={hex(keyPair.d)})")
from hashlib import sha512
hash = int.from_bytes(sha512(msg).digest(), byteorder='big')
signature = pow(hash, keyPair.d, keyPair.n)
print("Signature:", hex(signature))

Public key: (n=0xcac724613eb755d512b391fcecfa8b75c9f14d94872bf050894aca529e7da9edebe0db7266ceb22702e04fea4383851240afc7b63b04d390e5c333b18d378c8ed4c7b879cd77b71124c735a03e47ea20ceaaa48ee720b70220ae2f127046af1a84570b437b8204458aa365254587fd1ddded5833002877e65ffbae333a5fb6c1, e=0x10001)
Private key: (n=0xcac724613eb755d512b391fcecfa8b75c9f14d94872bf050894aca529e7da9edebe0db7266ceb22702e04fea4383851240afc7b63b04d390e5c333b18d378c8ed4c7b879cd77b71124c735a03e47ea20ceaaa48ee720b70220ae2f127046af1a84570b437b8204458aa365254587fd1ddded5833002877e65ffbae333a5fb6c1, d=0x1db7393996fb3ef4ce9cca140a965cc1922a6e7809a702bc8aa20f2c3455ae0553bfcbe8e3effe6381246611aa7b27911931a94ea72f79e18e0a21152fe0a814331b825e0858e974777d7d428606e3b7d37e5f0e3ce3e2358a0d561abc011f268443745010a299b7d96bcff335b281eb2ead48dd95eeaf52127cb72408236ac1)
Signature: 0x6ec6775ec98d8a6ba814e52640f06eab83c211fa8625200e88a4419302a9b0493fcaa05f8d5ba02638e2d7da6e9d51cfcb4bc513b6df2e86b74517ed65b9ae3cc2203457091c7f39ea01e27480fd5bac3da

In [ ]:
from Crypto.Random import random
from Crypto.PublicKey import DSA
from Crypto.Hash import SHA

# RSA verify signature
msg = b'A message for signing'
hash = int.from_bytes(sha512(msg).digest(), byteorder='big')
hashFromSignature = pow(signature, keyPair.e, keyPair.n)
print("Signature valid:", hash == hashFromSignature)

Signature valid: True


In [ ]:
!apt-get install gcc libgmp3-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libgmp-dev libgmpxx4ldbl
Suggested packages:
  gmp-doc libgmp10-doc libmpfr-dev
The following NEW packages will be installed:
  libgmp-dev libgmp3-dev libgmpxx4ldbl
0 upgraded, 3 newly installed, 0 to remove and 2 not upgraded.
Need to get 348 kB of archives.
After this operation, 1,725 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgmpxx4ldbl amd64 2:6.2.1+dfsg-3ubuntu1 [9,580 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgmp-dev amd64 2:6.2.1+dfsg-3ubuntu1 [337 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgmp3-dev amd64 2:6.2.1+dfsg-3ubuntu1 [1,818 B]
Fetched 348 kB in 0s (1,613 kB/s)
Selecting previously unselected package libgmpxx4ldbl:amd64.
(Reading database .

In [ ]:
!pip3 install ECPy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.4 MB/s eta 0:00:00


In [ ]:
from ecpy.curves     import Curve,Point
from ecpy.keys       import ECPublicKey, ECPrivateKey
from ecpy.ecdsa      import ECDSA

cv     = Curve.get_curve('secp256k1')
pu_key = ECPublicKey(Point(0x65d5b8bf9ab1801c9f168d4815994ad35f1dcb6ae6c7a1a303966b677b813b00,

                           0xe6b865e529b8ecbf71cf966e900477d49ced5846d7662dd2dd11ccd55c0aff7f,
                           cv))
pv_key = ECPrivateKey(0xfb26a4e75eec75544c0f44e937dcf5ee6355c7176600b9688c667e5c283b43c5,
                      cv)


signer = ECDSA()
sig    = signer.sign(b'01234567890123456789012345678912',pv_key)
assert(signer.verify(b'01234567890123456789012345678912',sig,pu_key))
print(sig)

b'0E\x02!\x00\xb2\x98;l\x14\x87\xbf\xe3\x9c\n\xeeR\xe2V\x99_\x0e\x0eG\x04\x7f6\xb0\xc6\x8dd\x98z\xe1g{f\x02 \rD\xae\x19\xda\x17\xc7\x10c\xe2J\xdc\xac\x12\x10\x91\xb0\x9a(f\xd0\xc6E\x8a\x90\x88\xc3\xf6j\xdb\xbd\xbf'


In [ ]:
from ecpy.curves     import Curve,Point

cv = Curve.get_curve('secp256k1')
P  = Point(0x65d5b8bf9ab1801c9f168d4815994ad35f1dcb6ae6c7a1a303966b677b813b00,
           0xe6b865e529b8ecbf71cf966e900477d49ced5846d7662dd2dd11ccd55c0aff7f,
           cv)
k  = 0xfb26a4e75eec75544c0f44e937dcf5ee6355c7176600b9688c667e5c283b43c5
Q  = k*P
R  = P+Q
print(k)
print(Q)
print(R)

113598803307006624577106489236173750803123281481805053449180702781422222132165
(0x7de099a1402a0a71881eb5f8c8469ae85431dd10943fa24981978062233394ce , 0x5723560f8b60f43464a37d2621ae907ba0d8d843d2989945e5e3c54614b142e7)
(0x43e5365b928c88c0cc940b4d0b5d1e7e81be5ee80a6d9c8f9b892e96530bf49b , 0x110ae6407e217fa372663c3e827fcaca1c05fdebd414034447393ee3c3deda96)


In [ ]:
from ecpy.curves import Curve
from ecpy.keys import ECPrivateKey
from ecpy.eddsa import EDDSA
import secrets, hashlib, binascii

curve = Curve.get_curve('Ed448')
signer = EDDSA(hashlib.shake_256, hash_len=114)
privKey = ECPrivateKey(secrets.randbits(57*8), curve)
pubKey = signer.get_public_key(privKey, hashlib.shake_256, hash_len=114)
print("Private key (57 bytes):", privKey)
print("Public key (compressed, 57 bytes): ",
      binascii.hexlify(curve.encode_point(pubKey.W)))
print("Public key (point): ", pubKey)

Private key (57 bytes): ECPrivateKey:
  d: eb5ace25df98894cb07a7ff74615a3cc21cbd29fbad923424957b3524b8125c17c71db2e165abbebea77940e1b88b227ca8d0e2b71a2e97f43
Public key (compressed, 57 bytes):  b'99c9b75a1f0aaf0d18ba27ee5f12bee89c01ae9bc410d3294583434e41782d35821a1ed5b841147de4c54a7bb36efcffde3a1dd082d0039f00'
Public key (point):  ECPublicKey:
  x: e1692cbec908f3b1be26ea6cff284c4d5b7d5db06990690ec0f11591a31933351c1a7b6ece3cf4ed2ce43ec178993ef504764258dac062c
  y: 9f03d082d01d3adefffc6eb37b4ac5e47d1441b8d51e1a82352d78414e43834529d310c49bae019ce8be125fee27ba180daf0a1f5ab7c999
